<a href="https://colab.research.google.com/github/Neoweikang/cvdl_10/blob/main/notebooks/demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Handwriting translator

Open a GPU runtime and choose **Runtime → Run all**. Allow the Drive mount, then choose a temporary demo password when prompted. Click **Open handwriting demo** and sign in as **presenter**. Keep the Colab session connected while presenting.

This launcher restores the English and Chinese OCR models from the verified 9 November run. Images and camera frames go through the password-protected Gradio link to your Colab runtime. **Try a bilingual sample** runs recognition on two handwritten lines from the project's demonstration page. Switch **Baseline** and **Trained** to read the same image with each model.


In [ ]:
# @title 1 · Restore the trained OCR models
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
project_notebook = Path('/content/drive/MyDrive/Colab Notebooks/dots_saved_runs.ipynb')
SAVED_RUN = '/content/drive/MyDrive/handwritten_translation_project/saved_training_runs/20261109T033914Z_187fd23d'  # @param {type:"string"}

os.environ['GRADIO_ANALYTICS_ENABLED'] = 'False'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet',
    'gradio==6.31.0', 'transformers==5.17.0', 'sentencepiece==0.2.1',
    'sacremoses==0.1.1', 'recognizers-text-number==1.0.2a2',
    'emoji==1.7.0', 'opencv-python-headless==4.11.0.86',
])

# Reuse the project's inference code. Section titles keep unrelated notebook edits
# and cell renumbering from changing which code the demo runs.
sections = {}
for cell in json.loads(project_notebook.read_text(encoding='utf-8'))['cells']:
    if cell['cell_type'] == 'code':
        source = ''.join(cell['source'])
        heading = source.partition('\n')[0].partition('·')[2].strip()
        if heading:
            sections[heading] = source

SESSION_MODE = 'restore'
SHOW_DEMO = False  # The presenter screen below supplies the interface.
CHECKPOINT_DIR = Path(tempfile.mkdtemp(prefix='cvdl-demo-'))
TRAINED_READY = set()
for heading in (
    'Shared OCR metrics and augmentation',
    'OCR checkpoint loading and inference',
    'Saved-run validation helpers',
    'Restore the explicitly selected completed run',
    'Marian model loading',
    'Timed translation decoding',
    'Reference-free review checks',
    'NLLB model and decoder',
    'Fixed translation routes',
    'Page detection and reading-order OCR',
    'Rule-aware line segmentation',
    'Paragraph translation with an explicit target',
    'Simple image and live OCR translator',
    'Mixed-page OCR translation',
):
    exec(compile(sections[heading], f'{project_notebook.name} — {heading}', 'exec'), globals())
print('Presenter models ready.')


In [ ]:
# @title 2 · Prepare the presentation screen
import gradio as gr

# The browser submits only an uploaded page or a stable, changed camera frame.
# Inference calls the same mixed-page function used in the project notebook.
def read_page(image: str, model: str) -> dict:
    return simple_ocr_translate(image, model)

def build_presenter():
    with gr.Blocks(title='Handwriting translator · CVDL', analytics_enabled=False, fill_width=True) as app:
        gr.HTML(html_template=PRESENTER_HTML, js_on_load=PRESENTER_JS, elem_id='reader-shell', container=False, padding=False, apply_default_css=False)
        gr.api(read_page, api_name='read_page', concurrency_limit=1)
    return app

PRESENTER_HTML = r"""<main id="ink-reader">
  <header class="mast">
    <div class="identity"><span class="seal" aria-hidden="true">文</span><span>CVDL <span class="mast-slash">/</span> Part C</span></div>
    <span class="language-pair">English <span aria-hidden="true">⇄</span> <span lang="zh-Hans">中文</span></span>
  </header>
  <section class="intro" aria-labelledby="app-name">
    <div><h1 id="app-name">Handwriting translator<span aria-hidden="true">.</span></h1>
      <p>Read and translate a handwritten page in English, Chinese, or both.</p></div>
    <fieldset class="model"><legend>OCR model</legend>
      <div class="seg"><label><input type="radio" name="ocr-model" value="baseline"><span>Baseline</span></label>
      <label><input type="radio" name="ocr-model" value="trained" checked><span>Trained</span></label></div>
    </fieldset>
  </section>
  <div class="workspace">
    <section class="source panel" aria-labelledby="source-heading">
      <div class="panel-head"><h2 id="source-heading">Source page</h2>
        <div class="modes" aria-label="Page source" role="group">
          <button data-mode="upload" aria-pressed="true">Upload</button><button data-mode="camera" aria-pressed="false">Live camera</button>
        </div>
      </div>
      <div class="stage" id="stage">
        <div class="landing" id="upload-landing">
          <span class="page-icon" aria-hidden="true"><i></i><i></i><i></i></span>
          <h3>Drop a handwritten page</h3>
          <p>JPG, PNG or WebP · up to 18 MB</p>
          <button class="primary" data-pick>Choose image <span aria-hidden="true">↑</span></button>
        </div>
        <div class="landing" id="camera-landing" hidden>
          <span class="camera-icon" aria-hidden="true"></span>
          <h3>Read from your camera</h3><p>Point it at a page, then hold still.</p>
          <button class="primary" id="start-camera">Start camera <span aria-hidden="true">→</span></button>
        </div>
        <img id="page-image" hidden alt="Selected handwriting">
        <video id="camera-feed" autoplay muted playsinline hidden aria-label="Live camera preview"></video>
      </div>
      <input type="file" id="file" accept="image/jpeg,image/png,image/webp" hidden>
      <div class="source-foot">
        <span class="caption" id="caption">Use horizontal writing in a single column.</span>
        <button class="link" id="change-page" data-pick hidden>Change image <span aria-hidden="true">→</span></button>
        <button class="link" id="restart-camera" hidden>Restart camera</button>
        <button class="link" id="stop-camera" hidden>Stop camera</button>
      </div>
      <button class="sample" id="sample"><img src="..." alt=""><span>Try a bilingual sample</span><span class="arrow" aria-hidden="true">→</span></button>
    </section>
    <section class="reading panel" aria-labelledby="reading-heading">
      <div class="panel-head"><h2 id="reading-heading">Reading</h2><span class="reading-meta" id="timing"></span></div>
      <div class="reading-inner">
        <section class="result-block"><div class="output-head"><h3>Literal transcription</h3><span id="language" class="script-tag" hidden></span></div>
          <div class="result-text" id="literal"><p class="placeholder">Recognized handwriting will appear here.</p></div>
        </section>
        <section class="result-block" id="translation-empty"><h3>Translation</h3><p class="placeholder" id="translation-hint">The detected language determines the translation.</p></section>
        <section class="result-block" id="en-block" hidden><h3>English translation</h3><div class="result-text" id="en" lang="en"></div></section>
        <section class="result-block" id="zh-block" hidden><h3>Chinese translation</h3><div class="result-text" id="zh" lang="zh-Hans"></div></section>
      </div>
    </section>
  </div>
  <div class="statusbar"><span class="dot" aria-hidden="true"></span><p id="status" role="status" aria-live="polite">Ready for a page.</p></div>
</main>
"""

PRESENTER_CSS = r"""@font-face {font-family:'DM Sans';font-style:normal;font-weight:100 1000;font-display:swap;src:local('DM Sans'),url(...) format('woff2')}
@font-face {font-family:'Baskervville';font-style:normal;font-weight:500;font-display:swap;src:local('Baskervville'),url(...) format('woff2')}
:root { color-scheme: light }
body, gradio-app, .gradio-container { background: #f7f5ef !important }
.gradio-container { max-width: none !important; width: 100% !important; padding: 0 !important; color-scheme: light }
.gradio-container:has(#ink-reader) > .main, #reader-shell .html-container { padding: 0 !important }
#reader-shell { border: 0 !important; padding: 0 !important; background: transparent !important }
#ink-reader {
  --ink: #242620; --muted: #4c4e45; --rule: #d5d3c7; --paper: #fdfcf8; --accent: #973d27;
  --sans: 'DM Sans', system-ui, sans-serif; --serif: Baskervville, Georgia, 'Times New Roman', serif;
  max-width: 1320px; margin: 0 auto; padding: 0 clamp(18px, 4.8vw, 72px) 36px;
  color: var(--ink); font: 16px/1.5 var(--sans); box-sizing: border-box; text-align: left;
}
#ink-reader *, #ink-reader *::before, #ink-reader *::after { box-sizing: border-box }
#ink-reader [hidden] { display: none !important }
#ink-reader :is(h1,h2,h3,p,fieldset,legend,ul) { margin: 0; padding: 0; color: inherit }
#ink-reader button { font: inherit; color: inherit; cursor: pointer; box-shadow: none; letter-spacing: 0 }
#ink-reader button:disabled, #ink-reader input:disabled + span { opacity: .53; cursor: wait }
#ink-reader button:focus-visible, #ink-reader input:focus-visible + span { outline: 3px solid #a4482d; outline-offset: 3px }
#ink-reader .mast { min-height: 72px; display: flex; align-items: center; justify-content: space-between; border-bottom: 1px solid #34372d }
#ink-reader .identity, #ink-reader .language-pair { display: inline-flex; align-items: center; gap: 12px; font-size: 14px; font-weight: 600 }
#ink-reader .seal { display: grid; place-items: center; width: 35px; height: 35px; background: var(--accent); color: white; font: 26px/1 'Noto Serif CJK SC','Songti SC','SimSun',serif; border-radius: 3px }
#ink-reader .mast-slash { color: #928e7e; margin: 0 4px }
#ink-reader .language-pair { font-size: 14px; font-weight: 500; gap: 7px }
#ink-reader .language-pair span:first-child { color: var(--accent) }
#ink-reader .intro { display: flex; flex-wrap: wrap; gap: 20px; justify-content: space-between; align-items: end; padding: clamp(28px,4vw,55px) 0 31px }
#ink-reader h1 { font: 500 clamp(42px,5.45vw,74px)/1.05 var(--serif); letter-spacing: -.055em }
#ink-reader h1 span { color: var(--accent) }
#ink-reader .intro p { color: var(--muted); margin-top: 32px; font-size: clamp(15px,1.35vw,18px) }
#ink-reader .model { border: 0; min-width: 0; flex-shrink: 0 }
#ink-reader legend { font-size: 13px; font-weight: 700; margin-bottom: 6px }
#ink-reader .seg { display: flex; padding: 3px; border: 1px solid var(--rule); border-radius: 7px; background: #eeeae0 }
#ink-reader .seg label { display: block; position: relative; cursor: pointer }
#ink-reader .seg input { position: absolute; opacity: 0; width: 1px; height: 1px }
#ink-reader .seg span { display: block; padding: 7px 13px; border-radius: 4px; font-size: 14px; font-weight: 600; color: #515247 }
#ink-reader .seg input:checked + span { color: white; background: #30372c }
#ink-reader .workspace { display: grid; grid-template-columns: minmax(0,1.06fr) minmax(0,1fr); gap: 22px; align-items: start }
#ink-reader .panel { min-width: 0; background: var(--paper); border: 1px solid var(--rule); border-top: 3px solid #424839; border-radius: 2px }
#ink-reader .panel-head { min-height: 72px; display: flex; align-items: center; justify-content: space-between; gap: 12px; padding: 13px 22px; border-bottom: 1px solid var(--rule) }
#ink-reader h2 { font: 500 clamp(26px,2.3vw,34px)/1.15 var(--serif); letter-spacing: -.045em }
#ink-reader .modes { display: flex; gap: 3px }
#ink-reader .modes button { padding: 7px 9px; border: 1px solid transparent; border-radius: 4px; background: transparent; color: var(--muted); font-size: 13px; font-weight: 600 }
#ink-reader .modes button[aria-pressed=true] { background: #eee9dc; border-color: #dad3c4; color: var(--ink) }
#ink-reader .stage { margin: 18px 18px 0; border: 1px dashed #bbb9a9; border-radius: 3px; position: relative; display: grid; place-items: center; min-height: clamp(237px,28vw,330px); overflow: hidden; background-color: #f5f3eb; background-image: linear-gradient(#dcd9cc52 1px,transparent 1px),linear-gradient(90deg,#dcd9cc52 1px,transparent 1px); background-size: 21px 21px }
#ink-reader .stage.drag { outline: 2px solid var(--accent); outline-offset: -3px; background-color: #f1e6d6 }
#ink-reader .stage.has-image { border-style: solid; border-color: var(--rule); background: #eeece3 }
#ink-reader .landing { padding: 26px 12px; text-align: center }
#ink-reader .landing h3 { font: 500 clamp(23px,2vw,29px)/1.2 var(--serif); letter-spacing: -.035em }
#ink-reader .landing p { margin: 6px 0 16px; color: var(--muted); font-size: 13px }
#ink-reader .page-icon { display: inline-flex; flex-direction: column; justify-content: center; gap: 5px; width: 32px; height: 40px; padding: 6px; margin-bottom: 12px; transform: rotate(-8deg); border: 1.5px solid var(--accent); box-shadow: 3px 3px 0 #e3ddcf; background: #faf7ee; border-radius: 2px }
#ink-reader .page-icon i { display: block; height: 1px; width: 100%; background: var(--accent) }
#ink-reader .page-icon i:last-child { width: 60% }
#ink-reader .camera-icon { display: inline-grid; place-items: center; width: 40px; height: 29px; border: 2px solid var(--accent); border-radius: 5px; margin: 12px auto 17px }
#ink-reader .camera-icon::before { content: ''; width: 13px; height: 13px; border-radius: 50%; border: 2px solid var(--accent) }
#ink-reader .primary { border: 0; border-radius: 4px; background: var(--accent); color: white; padding: 10px 16px; font-size: 14px; font-weight: 650 }
#ink-reader .primary span { margin-left: 9px; color: inherit }
#ink-reader .primary:hover { background: #7e321f }
#ink-reader #page-image, #ink-reader #camera-feed { display: block; width: 100%; height: auto; max-height: 60vh; object-fit: contain }
#ink-reader .stage.frozen #camera-feed { position: absolute; right: 8px; bottom: 8px; width: min(24%,118px); height: auto; max-height: 90px; border: 2px solid white; border-radius: 3px; box-shadow: 0 1px 6px #1d1d1d40 }
#ink-reader .source-foot { display: flex; justify-content: space-between; align-items: center; gap: 10px; padding: 12px 20px; min-height: 42px }
#ink-reader .caption, #ink-reader .reading-meta { color: var(--muted); font-size: 13px }
#ink-reader .link { font-size: 13px; font-weight: 650; background: transparent; border: 0; padding: 4px; color: var(--accent); white-space: nowrap }
#ink-reader .sample { width: 100%; display: flex; align-items: center; gap: 12px; padding: 12px 20px; border: 0; border-top: 1px solid var(--rule); background: transparent; text-align: left; font-size: 14px; font-weight: 600 }
#ink-reader .sample:hover { background: #f1eee5 }
#ink-reader .sample img { width: 67px; height: 32px; object-fit: cover; object-position: 8% 50%; border: 1px solid #c5c1b3; border-radius: 2px }
#ink-reader .arrow { margin-left: auto; font-size: 17px; color: var(--accent) }
#ink-reader .reading-inner { padding: 0 24px 14px }
#ink-reader .result-block { padding: 22px 0; min-height: 111px }
#ink-reader .result-block + .result-block:not([hidden]) { border-top: 1px solid var(--rule) }
#ink-reader .output-head { display: flex; align-items: center; justify-content: space-between; gap: 10px }
#ink-reader .result-block h3 { font: 650 16px/1.35 var(--sans) }
#ink-reader .result-text, #ink-reader .placeholder { margin-top: 10px; font: 500 clamp(17px,1.35vw,20px)/1.62 var(--sans); overflow-wrap: anywhere; white-space: pre-wrap }
#ink-reader .result-text p { font: inherit }
#ink-reader .placeholder, #ink-reader .result-text .placeholder { font: 400 15px/1.55 var(--sans); color: var(--muted) }
#ink-reader .script-tag { font: 550 12px/1.4 var(--sans); color: var(--accent) }
#ink-reader .line { display: grid; grid-template-columns: 23px minmax(0,1fr); gap: 7px; margin-top: 6px }
#ink-reader .num { color: var(--accent); font: 600 12px/1.7 var(--sans); border-right: 1px solid var(--rule); margin-top: 5px; align-self: start }
#ink-reader .missing { font-size: 14px; color: var(--muted); font-style: italic }
#ink-reader .statusbar { margin-top: 17px; display: flex; gap: 9px; align-items: baseline; color: var(--muted); font-size: 14px }
#ink-reader .dot { flex: 0 0 7px; height: 7px; border-radius: 50%; background: #727c60 }
#ink-reader[data-state=busy] .dot, #ink-reader[data-state=error] .dot { background: var(--accent) }
#ink-reader[data-state=error] #status { color: #9a301e }
@media(max-width: 800px) {
  #ink-reader .workspace { grid-template-columns: 1fr }
  #ink-reader .panel-head { min-height: 60px }
  #ink-reader .stage { min-height: 235px }
  #ink-reader .intro { padding-bottom: 24px }
}
@media(max-width: 450px) {
  #ink-reader { padding: 0 14px 24px }
  #ink-reader .mast { min-height: 55px }
  #ink-reader .intro { gap: 16px }
  #ink-reader .intro h1 { font-size: 40px }
  #ink-reader .panel-head, #ink-reader .sample, #ink-reader .source-foot { padding-left: 13px; padding-right: 13px }
  #ink-reader .stage { margin: 11px 11px 0 }
  #ink-reader .reading-inner { padding: 0 15px 10px }
}
"""

PRESENTER_JS = r"""const root = element.querySelector('#ink-reader');
const $ = id => root.querySelector('#' + id);
const modes = [...root.querySelectorAll('[data-mode]')];
const models = [...root.querySelectorAll('[name="ocr-model"]')];
const picks = [...root.querySelectorAll('[data-pick]')];
const stage = $('stage'), picture = $('page-image'), video = $('camera-feed');
const probe = document.createElement('canvas'), capture = document.createElement('canvas');
probe.width = 128; probe.height = 96;
const pixelsContext = probe.getContext('2d', {willReadFrequently: true});
let mode = 'upload', lastImage = null, stream = null, active = false;
let epoch = 0, busy = false, timer, previous = null, accepted = null, steady = 0;

function status(text, state = 'ready') {
  $('status').textContent = text; root.dataset.state = state;
}
function controls() {
  [...picks, ...models, ...modes, $('sample')].forEach(item => item.disabled = busy);
  modes.forEach(item => item.setAttribute('aria-pressed', String(item.dataset.mode === mode)));
  $('upload-landing').hidden = mode !== 'upload' || Boolean(lastImage);
  $('camera-landing').hidden = mode !== 'camera' || active || Boolean(lastImage);
  $('start-camera').disabled = busy;
  $('stop-camera').hidden = !active;
  $('restart-camera').hidden = mode !== 'camera' || active || !lastImage;
  $('restart-camera').disabled = busy;
  $('sample').hidden = mode !== 'upload';
  $('change-page').hidden = mode !== 'upload' || !lastImage;
  stage.classList.toggle('has-image', !picture.hidden || !video.hidden);
  stage.classList.toggle('frozen', !picture.hidden && !video.hidden);
  root.querySelector('.reading').setAttribute('aria-busy', String(busy));
}
function placeholder(node, message) {
  const p = document.createElement('p'); p.className = 'placeholder'; p.textContent = message;
  node.replaceChildren(p);
}
function clearReading() {
  placeholder($('literal'), 'Recognized handwriting will appear here.');
  $('en-block').hidden = $('zh-block').hidden = $('language').hidden = true;
  $('timing').textContent = ''; $('translation-empty').hidden = false;
  $('translation-hint').textContent = 'The detected language determines the translation.';
  $('en').replaceChildren(); $('zh').replaceChildren();
}
function releaseCamera() {
  clearTimeout(timer); active = false;
  if (stream) stream.getTracks().forEach(track => track.stop());
  stream = null; video.srcObject = null; video.hidden = true;
  previous = accepted = null; steady = 0;
}
function reset(nextMode) {
  epoch++; releaseCamera(); mode = nextMode; lastImage = null;
  picture.hidden = true; picture.removeAttribute('src'); $('file').value = '';
  clearReading(); controls();
  $('caption').textContent = nextMode === 'upload' ? 'Use horizontal writing in a single column.' : 'A stable, changed frame is read automatically.';
  status(nextMode === 'upload' ? 'Ready for a page.' : 'Ready to start the camera.');
}
function showImage(url, alt) {
  picture.src = url; picture.alt = alt; picture.hidden = false; controls();
}
function numbered(node, rows, textFor, emptyText) {
  node.replaceChildren();
  for (const row of rows) {
    const p = document.createElement('div'), n = document.createElement('span'), t = document.createElement('span');
    p.className = 'line'; n.className = 'num'; n.textContent = row.region_id;
    n.setAttribute('aria-label', 'Box ' + row.region_id);
    const text = textFor(row); t.textContent = text || emptyText;
    if (!text) t.className = 'missing';
    p.append(n,t); node.append(p);
  }
}
function render(result) {
  const region = result.regions || {}, lines = region.lines || [], translations = result.translations || {};
  if (region.image) {
    showImage(region.image, 'Processed snapshot with numbered boxes around detected handwriting');
    const count = region.boxes.length;
    $('caption').textContent = count + (count === 1 ? ' line' : ' lines') + ' detected · ' + region.width + ' × ' + region.height + ' px';
  }
  if (lines.length) numbered($('literal'), lines, row => row.raw_text, 'No text recognized');
  else if (result.text) $('literal').textContent = result.text;
  else placeholder($('literal'), result.error ? 'No transcription available.' : 'No readable handwriting detected.');
  if (result.language && (lines.length || result.text)) {
    $('language').textContent = result.language.split(' (')[0].replace('Mixed English/Chinese', 'English + Chinese');
    $('language').hidden = false;
  }
  $('translation-empty').hidden = Object.keys(translations).length > 0;
  $('translation-hint').textContent = 'No translation available for this page.';
  for (const [target, source] of [['en','zh'], ['zh','en']]) {
    if (!Object.hasOwn(translations, target)) continue;
    $(target + '-block').hidden = false;
    if (result.mixed && lines.length) {
      // Preserve the project display: a bilingual page shows each translated box once per target.
      const selected = lines.filter(row => row.script === source || row.script === 'mixed');
      if (selected.length) numbered($(target), selected, row => row.translations[target], 'No translation returned');
      else placeholder($(target), 'No lines required translation into this language.');
    } else $(target).textContent = translations[target] || 'No translation returned.';
  }
  if (typeof result.seconds === 'number') $('timing').textContent = result.seconds + ' s';
}
// A queued request keeps the browser connected while a cold model downloads.
async function requestPage(image, model) {
  const endpoint = new URL('gradio_api/call/read_page', document.baseURI);
  const job = await fetch(endpoint, {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify({data:[image, model]})});
  if (!job.ok) throw new Error(job.status === 401 ? 'Sign in again to continue.' : 'The Colab runtime could not start this request.');
  const {event_id} = await job.json();
  return new Promise((resolve, reject) => {
    const events = new EventSource(endpoint.href + '/' + encodeURIComponent(event_id));
    events.addEventListener('complete', event => {
      events.close();
      try { resolve(JSON.parse(event.data)[0]); } catch (error) { reject(error); }
    });
    events.onerror = () => { events.close(); reject(new Error('Lost connection to the model. Check the Colab runtime and try again.')); };
  });
}
async function read(image, ticket) {
  if (busy || ticket !== epoch) return;
  const choice = models.find(item => item.checked).value;
  busy = true; clearReading(); showImage(image, 'Source image being read');
  $('caption').textContent = 'Reading the source page…';
  status('Reading and translating… The first page can take longer while models load.', 'busy');
  try {
    const result = await requestPage(image, choice);
    if (ticket !== epoch) return;
    if (!result || typeof result !== 'object') throw new Error('The runtime returned no result.');
    render(result);
    if (result.error) throw new Error(result.error);
    const modelName = choice === 'trained' ? 'Trained' : 'Baseline';
    status(modelName + ' model finished.' + (active ? ' Move to the next page and hold still.' : ''));
  } catch (error) {
    if (ticket === epoch) {
      if (active) releaseCamera();
      $('caption').textContent = picture.alt.startsWith('Processed') ? $('caption').textContent : 'Source image';
      if (!root.querySelector('#literal .line') && $('literal').querySelector('.placeholder')) {
        placeholder($('literal'), 'No transcription available.');
      }
      $('translation-hint').textContent = 'No translation available.';
      status(error.message || 'Connection lost. Check the Colab runtime and try again.', 'error');
    }
  } finally { busy = false; controls(); }
}
async function openFile(file) {
  if (!file || busy) return;
  reset('upload');
  if (file.size > 18000000) { status('Choose an image under 18 MB.', 'error'); return; }
  if (!['image/jpeg','image/png','image/webp'].includes(file.type)) {
    status('Choose a JPG, PNG or WebP image.', 'error'); return;
  }
  const ticket = epoch, reader = new FileReader();
  reader.onerror = () => { if (ticket === epoch) status('This file could not be read.', 'error'); };
  reader.onload = () => {
    if (ticket !== epoch) return;
    lastImage = reader.result; read(lastImage, ticket);
  };
  reader.readAsDataURL(file);
}
// The original live detector compares 8 × 6 tiles so a new word can trigger OCR.
function difference(a, b) {
  if (!a || !b) return Infinity;
  let total = 0; const tiles = new Float32Array(48);
  for (let i = 0; i < a.length; i += 4) {
    const delta = (Math.abs(a[i]-b[i]) + Math.abs(a[i+1]-b[i+1]) + Math.abs(a[i+2]-b[i+2])) / 3;
    const pixel = i / 4, tile = Math.floor((pixel % 128) / 16) + 8 * Math.floor(pixel / 128 / 16);
    total += delta; tiles[tile] += delta;
  }
  return Math.max(total / (a.length / 4), Math.max(...tiles) / 256 * .75);
}
async function tick(ticket) {
  if (!active || ticket !== epoch) return;
  try {
    if (!busy && video.readyState >= 2 && video.videoWidth) {
      pixelsContext.drawImage(video, 0, 0, 128, 96);
      const pixels = pixelsContext.getImageData(0, 0, 128, 96).data.slice();
      steady = difference(pixels, previous) < 4 ? steady + 1 : 0; previous = pixels;
      if (steady >= 2 && difference(pixels, accepted) > 2.5) {
        const scale = Math.min(1, 1400 / Math.max(video.videoWidth, video.videoHeight));
        capture.width = Math.max(2, Math.round(video.videoWidth * scale));
        capture.height = Math.max(2, Math.round(video.videoHeight * scale));
        capture.getContext('2d').drawImage(video, 0, 0, capture.width, capture.height);
        accepted = pixels; steady = 0; lastImage = capture.toDataURL('image/jpeg', .9);
        await read(lastImage, ticket);
      }
    }
  } catch (error) {
    if (ticket === epoch) { releaseCamera(); controls(); status('Camera stopped. ' + error.message, 'error'); }
  }
  if (active && ticket === epoch) timer = setTimeout(() => tick(ticket), 600);
}
async function startCamera() {
  if (busy || active) return;
  reset('camera'); active = true; const ticket = epoch; controls();
  status('Allow camera access, then hold your page still.');
  try {
    const acquired = await navigator.mediaDevices.getUserMedia({video:{facingMode:{ideal:'environment'}}, audio:false});
    if (!active || ticket !== epoch) { acquired.getTracks().forEach(track => track.stop()); return; }
    stream = acquired; video.srcObject = stream; video.hidden = false; controls();
    await video.play();
    if (active && ticket === epoch) timer = setTimeout(() => tick(ticket), 600);
  } catch (error) {
    if (ticket === epoch) { releaseCamera(); controls(); status('Camera unavailable. ' + error.message, 'error'); }
  }
}
modes.forEach(button => button.onclick = () => { if (mode !== button.dataset.mode) reset(button.dataset.mode); });
picks.forEach(button => button.onclick = () => $('file').click());
$('file').onchange = () => openFile($('file').files[0]);
stage.ondragover = event => { event.preventDefault(); if (!busy && mode === 'upload') stage.classList.add('drag'); };
stage.ondragleave = () => stage.classList.remove('drag');
stage.ondrop = event => {
  event.preventDefault(); stage.classList.remove('drag');
  if (mode === 'upload') openFile(event.dataTransfer.files[0]);
};
$('sample').onclick = () => {
  reset('upload'); lastImage = $('sample').querySelector('img').src; read(lastImage, epoch);
};
models.forEach(item => item.onchange = () => { if (lastImage) read(lastImage, epoch); });
$('start-camera').onclick = $('restart-camera').onclick = startCamera;
$('stop-camera').onclick = () => {
  epoch++; releaseCamera(); controls();
  if (busy) { $('caption').textContent = 'Captured frame'; clearReading(); }
  status('Camera stopped.');
};
window.addEventListener('pagehide', () => { epoch++; releaseCamera(); }, {once:true});
controls();
"""

# Bundled fonts
# Copyright 2014 The DM Sans Project Authors (https://github.com/googlefonts/dm-fonts)
# Copyright 2018 The Baskervville Project Authors (https://github.com/anrt-type/ANRT-Baskervville)
#
# This Font Software is licensed under the SIL Open Font License, Version 1.1.
# This license is copied below, and is also available with a FAQ at:
# https://scripts.sil.org/OFL
#
#
# -----------------------------------------------------------
# SIL OPEN FONT LICENSE Version 1.1 - 26 February 2007
# -----------------------------------------------------------
#
# PREAMBLE
# The goals of the Open Font License (OFL) are to stimulate worldwide
# development of collaborative font projects, to support the font creation
# efforts of academic and linguistic communities, and to provide a free and
# open framework in which fonts may be shared and improved in partnership
# with others.
#
# The OFL allows the licensed fonts to be used, studied, modified and
# redistributed freely as long as they are not sold by themselves. The
# fonts, including any derivative works, can be bundled, embedded,
# redistributed and/or sold with any software provided that any reserved
# names are not used by derivative works. The fonts and derivatives,
# however, cannot be released under any other type of license. The
# requirement for fonts to remain under this license does not apply
# to any document created using the fonts or their derivatives.
#
# DEFINITIONS
# "Font Software" refers to the set of files released by the Copyright
# Holder(s) under this license and clearly marked as such. This may
# include source files, build scripts and documentation.
#
# "Reserved Font Name" refers to any names specified as such after the
# copyright statement(s).
#
# "Original Version" refers to the collection of Font Software components as
# distributed by the Copyright Holder(s).
#
# "Modified Version" refers to any derivative made by adding to, deleting,
# or substituting -- in part or in whole -- any of the components of the
# Original Version, by changing formats or by porting the Font Software to a
# new environment.
#
# "Author" refers to any designer, engineer, programmer, technical
# writer or other person who contributed to the Font Software.
#
# PERMISSION & CONDITIONS
# Permission is hereby granted, free of charge, to any person obtaining
# a copy of the Font Software, to use, study, copy, merge, embed, modify,
# redistribute, and sell modified and unmodified copies of the Font
# Software, subject to the following conditions:
#
# 1) Neither the Font Software nor any of its individual components,
# in Original or Modified Versions, may be sold by itself.
#
# 2) Original or Modified Versions of the Font Software may be bundled,
# redistributed and/or sold with any software, provided that each copy
# contains the above copyright notice and this license. These can be
# included either as stand-alone text files, human-readable headers or
# in the appropriate machine-readable metadata fields within text or
# binary files as long as those fields can be easily viewed by the user.
#
# 3) No Modified Version of the Font Software may use the Reserved Font
# Name(s) unless explicit written permission is granted by the corresponding
# Copyright Holder. This restriction only applies to the primary font name as
# presented to the users.
#
# 4) The name(s) of the Copyright Holder(s) or the Author(s) of the Font
# Software shall not be used to promote, endorse or advertise any
# Modified Version, except to acknowledge the contribution(s) of the
# Copyright Holder(s) and the Author(s) or with their explicit written
# permission.
#
# 5) The Font Software, modified or unmodified, in part or in whole,
# must be distributed entirely under this license, and must not be
# distributed under any other license. The requirement for fonts to
# remain under this license does not apply to any document created
# using the Font Software.
#
# TERMINATION
# This license becomes null and void if any of the above conditions are
# not met.
#
# DISCLAIMER
# THE FONT SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND,
# EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO ANY WARRANTIES OF
# MERCHANTABILITY, FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT
# OF COPYRIGHT, PATENT, TRADEMARK, OR OTHER RIGHT. IN NO EVENT SHALL THE
# COPYRIGHT HOLDER BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER LIABILITY,
# INCLUDING ANY GENERAL, SPECIAL, INDIRECT, INCIDENTAL, OR CONSEQUENTIAL
# DAMAGES, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING
# FROM, OUT OF THE USE OR INABILITY TO USE THE FONT SOFTWARE OR FROM
# OTHER DEALINGS IN THE FONT SOFTWARE.


In [ ]:
# @title 3 · Open the presenter link
from getpass import getpass
from IPython.display import HTML, display
from html import escape

password = getpass('Choose a password for the presenter account › ')
if not password.strip():
    raise ValueError('Enter a password for the presenter link, then rerun this cell.')
if 'presenter_app' in globals():
    presenter_app.close()
presenter_app = build_presenter()
_, _, presenter_url = presenter_app.launch(
    share=True, inline=False, prevent_thread_lock=True, quiet=True,
    auth=('presenter', password), auth_message='Sign in as presenter with the password from your Colab session.',
    css=PRESENTER_CSS, footer_links=[], blocked_paths=['/content/drive', str(CHECKPOINT_DIR)],
)
del password
if not presenter_url:
    raise RuntimeError('The presenter link could not be created. Rerun this cell to reconnect.')
display(HTML(f'<a href="{escape(presenter_url, quote=True)}" target="_blank" rel="noopener">Open handwriting demo ↗</a>'))
